# Stage 2 Notebook 26 - Exp2U Lane-only diagnostic (zero detection loss)

**Why this exists.** After 14 attempts to fix the lane head architecture (Exp2G through Exp2T), `decoded_f1` has been frozen in [0.026, 0.040] for SIX query-style experiments. Meanwhile across ALL Exp2 runs:
- `mAP50 = 0.003-0.005` -- detection has NEVER converged.
- `lambda_lane = 0.05-0.17` from grad_norm calibration -- detection's gradient is being weighted 6-20x more than lane in the joint loss because det_loss is much larger.
- DETR with 100 queries needs 50+ epochs to converge. We train for 10. So detection produces large but non-converging gradient flow into the shared backbone for the entire training run.

Hypothesis: **detection's broken-but-loud gradient has been sabotaging lane training**. The shared backbone is being pulled toward features that satisfy a non-converging set-prediction task, with 6-20x the gradient magnitude of the lane task. Lane features get reset every batch by detection's backbone updates.

Exp2U is the decisive diagnostic: same lane head as Exp2P (proven cls=0.65), but **zero all detection loss weights** and **fix lambda_lane=1.0** (no grad_norm). Detection head still runs forward (architecture identical) but contributes nothing to backprop. If detection has been the saboteur, lane metrics should jump dramatically.

Decision tree at epoch 10:
- `decoded_f1 >= 0.15` (4x current best): detection IS the saboteur. Stage 3 path: fix detection first (Exp2V tests grid head as alternative), then re-enable joint training.
- `decoded_f1 ~ 0.04` (no change): detection is NOT the issue. The lane impasse comes from something deeper -- training duration (10 vs CLRKDNet's 70 epochs), image resolution (384x640 vs 590x1640), or dataset size.

Single-knob change vs Exp2P: `loss.det.* = 0`, `lambda_lane: 1.0`, `lambda_mode: fixed`. All architecture identical.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp21_rmt_gca_lane_only_diagnostic_joint_smoke.log
OK exp21_rmt_gca_lane_only_diagnostic_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=2.2600 det_loss=0.0000 grad_cos=0.0000 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5007621645927429, 'gate/lane_mean': 0.4975205659866333, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp21_rmt_gca_lane_only_diagnostic_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp21_rmt_gca_lane_only_diagnostic_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml --curve-tar /content/drive/MyDrive

0

## What to watch in Exp2U training

Reference Exp2P (queries, joint detection): `decoded_f1=0.026`, `matched_iou=0.13`, `oracle_f1=0.07`.

Strong signals that detection has been sabotaging lane:
- **`val/lane/decoded_f1 >= 0.15`** (4-6x Exp2P). The metric that matters.
- **`val/matched_line_iou >= 0.30`** (vs Exp2P's 0.13). Geometry should improve dramatically once the backbone isn't being pulled by detection.
- **`val/lane/decoded_oracle_f1 >= 0.20`** (vs Exp2P's 0.07). Geometry ceiling rises with backbone stability.
- `val_lane_f1 >= 0.65` (cls task unchanged).

Weak / no signal:
- decoded_f1 stays at ~0.03: detection is not the issue. Pivot to: extended training (30+ epochs), higher resolution (720x1280 if memory allows), or KD from a CLRKDNet teacher.
- Geometry improves but cls regresses: lane-only training disrupted the cls/geometry balance; rebalance w_cls.

After short10, run NB08 to plot Exp2P vs Exp2U side-by-side. The matched_iou and decoded_f1 deltas tell us exactly how much detection has been costing us.